In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from artifact_core.libs.implementation.tabular.cdf.overlaid_plotter import OverlaidCDFPlotter
from artifact_core.libs.implementation.tabular.cdf.plotter import CDFPlotter
from artifact_core.libs.implementation.tabular.pairwise_correlation.calculator import (
    CategoricalAssociationType,
    ContinuousAssociationType,
)
from artifact_core.libs.implementation.tabular.pairwise_correlation.plotter import (
    PairwiseCorrelationHeatmapPlotter,
)
from artifact_core.libs.implementation.tabular.pdf.overlaid_plotter import OverlaidPDFPlotter
from artifact_core.libs.implementation.tabular.pdf.plotter import PDFPlotter


## 1. Generate Challenging Datasets

We'll create datasets that would typically cause plotting issues:
- **Many categorical features** with high cardinality
- **Many continuous features** for subplot layouts
- **Large correlation matrices** for heatmaps


In [ ]:
def generate_challenging_dataset(n_samples: int = 1000) -> tuple:
    """Generate a dataset designed to stress-test our plotting autoscaling."""
    np.random.seed(42)  # For reproducibility

    data = {}

    # === CONTINUOUS FEATURES (80 features) ===
    print("🔢 Generating 80 continuous features...")
    continuous_features = []
    for i in range(80):
        feature_name = f"continuous_feature_{i + 1:02d}"
        # Mix of different distributions
        if i % 3 == 0:
            data[feature_name] = np.random.normal(50, 15, n_samples)
        elif i % 3 == 1:
            data[feature_name] = np.random.exponential(2, n_samples)
        else:
            data[feature_name] = np.random.gamma(2, 2, n_samples)
        continuous_features.append(feature_name)

    # === CATEGORICAL FEATURES WITH MANY CATEGORIES ===
    categorical_features = []
    cat_unique_map = {}

    # High-cardinality categorical (30 categories)
    super_high_card_categories = [f"Category_{i + 1:02d}" for i in range(500)]
    data["super_high_cardinality_cat"] = np.random.choice(super_high_card_categories, n_samples)
    categorical_features.append("super_high_cardinality_cat")
    cat_unique_map["super_high_cardinality_cat"] = super_high_card_categories

    # High-cardinality categorical (30 categories)
    high_card_categories = [f"Category_{i + 1:02d}" for i in range(100)]
    data["high_cardinality_cat"] = np.random.choice(high_card_categories, n_samples)
    categorical_features.append("high_cardinality_cat")
    cat_unique_map["high_cardinality_cat"] = high_card_categories

    # Medium-cardinality categorical (15 categories)
    med_card_categories = [f"Type_{chr(65 + i)}" for i in range(30)]  # A-O
    data["medium_cardinality_cat"] = np.random.choice(med_card_categories, n_samples)
    categorical_features.append("medium_cardinality_cat")
    cat_unique_map["medium_cardinality_cat"] = med_card_categories

    # Another high-cardinality (25 categories)
    product_categories = [f"Product_{i + 1:03d}" for i in range(25)]
    data["product_category"] = np.random.choice(product_categories, n_samples)
    categorical_features.append("product_category")
    cat_unique_map["product_category"] = product_categories

    # Region categories (20 categories)
    regions = [f"Region_{i + 1:02d}" for i in range(20)]
    data["region"] = np.random.choice(regions, n_samples)
    categorical_features.append("region")
    cat_unique_map["region"] = regions

    # Create DataFrame
    df = pd.DataFrame(data)

    # Feature ordering (mix continuous and categorical)
    features_order = continuous_features + categorical_features

    print(f"📊 Generated dataset with {len(df)} rows:")
    print(f"   🔢 {len(continuous_features)} continuous features")
    print(f"   🏷️  {len(categorical_features)} categorical features")
    print(f"   📈 Category counts: {[len(cats) for cats in cat_unique_map.values()]}")

    return df, features_order, continuous_features, categorical_features, cat_unique_map


# Generate real and synthetic datasets
print("🎲 Generating challenging datasets...")
real_data, features_order, continuous_features, categorical_features, cat_unique_map = (
    generate_challenging_dataset(1000)
)

# Create a "synthetic" version with slight variations
synthetic_data = real_data.copy()
for col in continuous_features:
    # Add some noise and shift to make synthetic different
    synthetic_data[col] = real_data[col] + np.random.normal(
        0, 0.1 * real_data[col].std(), len(real_data)
    )

for col in categorical_features:
    # Randomly reassign ~10% of categorical values
    mask = np.random.random(len(synthetic_data)) < 0.1
    if mask.sum() > 0:
        synthetic_data.loc[mask, col] = np.random.choice(cat_unique_map[col], mask.sum())

print("✅ Datasets ready for autoscaling demonstration!")


## 3. PDF Plots with Categorical Autoscaling

Test categorical autoscaling with high-cardinality categorical features. Without autoscaling, these would be unreadable.


In [ ]:
pdf_plot = PDFPlotter.get_pdf_plot(
    dataset=real_data,
    ls_features_order=features_order,
    ls_cts_features=continuous_features,
    ls_cat_features=categorical_features,
    cat_unique_map=cat_unique_map,
)

pdf_plot

In [ ]:
overlaid_pdf_plot = OverlaidPDFPlotter.get_overlaid_pdf_plot(
    dataset_real=real_data,
    dataset_synthetic=synthetic_data,
    ls_features_order=features_order,
    ls_cts_features=continuous_features,
    ls_cat_features=categorical_features,
    cat_unique_map=cat_unique_map,
)

overlaid_pdf_plot

In [ ]:
overlaid_pdf_plot_collection = OverlaidPDFPlotter.get_overlaid_pdf_plot_collection(
    dataset_real=real_data,
    dataset_synthetic=synthetic_data,
    ls_features_order=features_order,
    ls_cts_features=continuous_features,
    ls_cat_features=categorical_features,
    cat_unique_map=cat_unique_map,
)

In [ ]:
overlaid_pdf_plot_collection["super_high_cardinality_cat"]

## 4. CDF Plots with Subplot Autoscaling

Test subplot autoscaling with many continuous features. Without autoscaling, individual plots would be too small to read.


In [ ]:
cdf_plot = CDFPlotter.get_cdf_plot(dataset=real_data, ls_cts_features=continuous_features)

cdf_plot

In [ ]:
overlaid_cdf_plot = OverlaidCDFPlotter.get_overlaid_cdf_plot(
    dataset_real=real_data, dataset_synthetic=synthetic_data, ls_cts_features=continuous_features
)

overlaid_cdf_plot

## 5. Correlation Heatmaps with Grid Autoscaling

Test grid autoscaling with large correlation matrices. Without autoscaling, annotations would be unreadable.


In [ ]:
correlation_plot = PairwiseCorrelationHeatmapPlotter.get_combined_correlation_plot(
    categorical_correlation_type=CategoricalAssociationType.CRAMERS_V,
    continuous_correlation_type=ContinuousAssociationType.PEARSON,
    dataset_real=real_data,
    dataset_synthetic=synthetic_data,
    ls_cat_features=categorical_features,
)

correlation_plot

In [ ]:
correlation_plot = PairwiseCorrelationHeatmapPlotter.get_correlation_difference_heatmap(
    categorical_correlation_type=CategoricalAssociationType.CRAMERS_V,
    continuous_correlation_type=ContinuousAssociationType.PEARSON,
    dataset_real=real_data,
    dataset_synthetic=synthetic_data,
    ls_cat_features=categorical_features,
)

correlation_plot